In [29]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [31]:
df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
pd.set_option("display.max_columns",None)

print (df.head(10))
print (df.info())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   
5  9305-CDSKC  Female              0      No         No       8          Yes   
6  1452-KIOVK    Male              0      No        Yes      22          Yes   
7  6713-OKOMC  Female              0      No         No      10           No   
8  7892-POOKP  Female              0     Yes         No      28          Yes   
9  6388-TABGU    Male              0      No        Yes      62          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No  

In [32]:
def feature_engineering(x_train , x_test):
    encoder = OneHotEncoder(sparse_output = False , handle_unknown ="ignore").set_output(transform="pandas")
    x_train_encoded = encoder.fit_transform(x_train)
    x_test_encoded = encoder.transform(x_test)
    return x_train_encoded , x_test_encoded

In [33]:
def feature_scaling(x_train,x_test):
    std = StandardScaler()
    x_train_scaled = pd.DataFrame(std.fit_transform(x_train),columns=x_train.columns,index=x_train.index)
    x_test_scaled = pd.DataFrame(std.transform(x_test),columns=x_test.columns,index=x_test.index)

    return x_train_scaled,x_test_scaled


In [34]:
yes_no_cols = [col for col in df.columns if set(df[col].dropna().unique()) == {'Yes', 'No'}]
print(yes_no_cols)

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


In [35]:
def Classification(model,x_train,y_train,x_test,y_test):
    for name,model in model.items():
        model.fit(x_train,y_train)
        pred = model.predict(x_test)
        train_score = model.score(x_train,y_train)
    
        train_acc = model.score(x_train, y_train)
        test_acc = accuracy_score(y_test, pred)
        precision = precision_score(y_test, pred)
        recall = recall_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        
        print(f"--- {name} ---")
        print(f"Train Accuracy : {train_acc:.4f}")
        print(f"Test Accuracy  : {test_acc:.4f}")
        print(f"Precision      : {precision:.4f}")
        print(f"Recall         : {recall:.4f}")
        print(f"F1 Score       : {f1:.4f}")
        print(f"Confusion Matrix:\n{confusion_matrix(y_test, pred)}\n")


In [38]:
   
x_train,x_test, y_train , y_test = train_test_split (df.drop(columns=["Churn","customerID"]),df["Churn"],random_state=42)

    
x_train_numeric = x_train.select_dtypes(include=['int64','float64'])
x_train_variable = x_train.select_dtypes(include=['object'])

x_test_numeric = x_test.select_dtypes(include=['int64','float64'])
x_test_variable = x_test.select_dtypes(include=['object'])

x_train_variable_encoded,x_test_variable_encoded = feature_engineering(x_train_variable,x_test_variable)

x_train_encoded = pd.concat([x_train_variable_encoded,x_train_numeric],axis=1)
x_test_encoded = pd.concat([x_test_variable_encoded,x_test_numeric],axis=1)

print(x_train_encoded.shape)
# print(df.corr(numeric_only=True)['Churn'].sort_values(ascending=False))

x_train_scaled,x_test_scaled = feature_scaling(x_train_encoded,x_test_encoded)
print(x_train_scaled.isnull().sum())


(5282, 41)
gender_Female                              0
gender_Male                                0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBackup_No                            0
OnlineBackup_No internet service           0
OnlineBackup_Yes                           0
DeviceProtection_No                        0
DeviceProtection_No internet service       0
DeviceProtection_Yes                       0
TechSupport_No                             0
TechSupport_No internet service            0
TechSupport_Yes                            0
StreamingTV_No                             0
StreamingTV_No internet service            0

In [37]:
model = {
    "Logistic Regression":LogisticRegression()}
Classification(model,x_train_scaled,y_train,x_test_scaled,y_test)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values